# Zerlaut parameter exploration

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
from copy import deepcopy

In [ ]:
from aux.environment import resolve_directory
from aux.parameters import bounds_from_default

**Console**

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print
pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Auxiliar Functions

In [ ]:
def remove_unsampled_parameters(params, names) -> dict:
    result = deepcopy(params)
    for key in names:
        result.pop(key, None) # Evita el error si no se encuentra
    return result

In [ ]:
def count_parameters(params):
    return sum(
        len(v['default']) if isinstance(v['default'], list) else 1 
        for v in params.values()
    )

## Parameter sampling


In [ ]:
CONFIG_PATH = WORK_DIR / 'config' / 'zerlaut' / 'params_raw.json'
pprint(f'Parameter configuration: {CONFIG_PATH}')

In [ ]:
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PARAMS = json.load(config_file)

SAMPLED_PARAMS, FIX_PARAMS = deepcopy(PARAMS), {}

pprint(f'Loaded {count_parameters(PARAMS)}', str(list(PARAMS.keys())))

**Common processing**

1. **Remove unsampled parameters**

In [ ]:
SAMPLED_PARAMS = remove_unsampled_parameters(SAMPLED_PARAMS, [
    "external_input_ex_ex", "external_input_ex_in", "external_input_in_ex", "external_input_in_in" #,S_i
])

2. **Add undefined bounds**

In [ ]:
SAMPLED_PARAMS = bounds_from_default(SAMPLED_PARAMS, names=["P_e", "P_i"], margin=0.10)

for param in (SAMPLED_PARAMS['P_e'], SAMPLED_PARAMS['P_i']):
    p = param['default'][0]
    param['min'][0], param['max'][0] = p - 0.05 * p, p + 0.05 * p

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo


3. **Adjust noise**

In [ ]:
## Reduce noise
SAMPLED_PARAMS['tau_OU'] |= {'min': 0.05, 'max': 4, 'default': 2}
SAMPLED_PARAMS['weight_noise'] |= {'min': 0.00001, 'max': 0.0004, 'default': 0.1}

# NOTE: Puede merecer la pena usar una distribución beta para el muetreo

### Parameter configuration 1

* Add beta distiribution to noise parameters and P_e, P_i

In [ ]:

pprint(f'Loaded {count_parameters(PARAM_CONFIG1)}', str(list(PARAM_CONFIG1.keys())))

**Add undefined bounds**

## Parameter test

### Common settings

In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"
MODEL_NAME = "ZerlautAdaptationFirstOrder" # Solo por tiempo!!!!

BASE_PARAMS = bounds_from_default(PARAMS, ["P_e", "P_i"], margin=0.10)
# BASE_PARAMS = add_truncnorm_config_safe(BASE_PARAMS, std_factor=0.2)
# BASE_PARAMS = add_beta_config_safe(BASE_PARAMS, lambda_=2.0)



INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"} 

pprint('New bounded parameters:', str({k: v for k, v in BASE_PARAMS.items() if k in ["P_e", "P_i"]}))

In [ ]:
# 1) Entradas de la simulación.

# 1.0) Salida de la simulación.
# OUTPUT_STATES = None
# OUTPUT_STATES = ['E', 'I', 'W_e', 'W_i', 'ou_drift']
OUTPUT_STATES = ['E']

# Solo requerido para tareas como plotting.
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}

STIMULUS_UNITS = {
    "external_input_ex_ex": "kHz",
    "external_input_ex_in": "kHz",
    "external_input_in_ex": "kHz",
    "external_input_in_in": "kHz",
}


In [ ]:
# 1.1) Parámetros de la simulación.
DURATION_MS = 2000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.5             # Paso de integración en milisegundos.

zero_nsig_ = [0] * len(svar := get_model_class(MODEL_NAME).state_variables)
nsig_ = [int(i == svar.index("ou_drift")) for i in range(len(svar))]


INTEGRATOR_NAME = 'HeunStochastic'  
NOISE ={
    "name": "Additive", "params": {"nsig": nsig_} # [0,0,0,0,1] in first order model
}

NO_NOISE ={
    "name": "Additive", "params": {"nsig": zero_nsig_} # [0,0,0,0,0] in first order
}

NOISE_SEED = 42

In [ ]:
# 1.2) Parámetros de la entrada. En este caso un pulso.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.001,        # Frecuencia de disparo basal 1Hz
            "pulse_amplitude_khz": 0.01,  # Amplitud del pulso 10 Hz
            "pulse_onset_ms": 200.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 160.0,   # Duración del pulso
            "pulse_period_ms": 320.0,     # Periodo del pulso
        },
    },   
    "external_input_in_ex": {"external_input_ex_ex": 1}, 
}


STIMULUS_EX = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.005,        # Frecuencia de disparo basal
            "pulse_amplitude_khz": 0.04,  # Amplitud del pulso
            "pulse_onset_ms": 200.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 160.0,   # Duración del pulso
            "pulse_period_ms": 320.0,     # Periodo del pulso
        },
    },   
}




In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)
time_ms, stimulus_ex = build_stimulus(STIMULUS_EX, DURATION_MS, DT_MS)

# Plotting

fig, ax = plt.subplots(figsize=(10, 3))

for key, values in stimulus:
    ax.plot(time_ms, values, label=key)

ax.set_title("Stimulation Signals")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Stimulus (KHz)")
ax.legend()
plt.show()

In [ ]:
# Sample parameters
SAMPLES_PER_AXIS = 4
GRID_SPACING = 'linear'  # Set to 'log' for logarithmic spacing.
SAMPLING_SEED = 42
FIRST_PARAM_INDEX = 0
SECOND_PARAM_INDEX = 0


### Noise

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Noise'] #+["whatever"]

pprint(f'Noise len: {len(GROUPS["Noise"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0} # Para reproducir los experimentos de otras cuadernos.
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

#EXPLORATION_PARAMS["weight_noise"]["min"] = 0.0001
#EXPLORATION_PARAMS["weight_noise"]["max"] = 1.0
#EXPLORATION_PARAMS["tau_OU"]["min"] = 0.4

# Modificar los parámetros de muestro
# EXPLORATION_PARAMS |= add_beta_config_safe(
#     {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE},
#     lambda_=5
# )

GRID_SPACING = 'log' # 'linear'

USE_NOISE = True
USE_STIMULUS_EX = False

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {BASE_PARAMS[p]["default"]}')
    pprint(f'  Min: {BASE_PARAMS[p]["min"]}')
    pprint(f'  Max: {BASE_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))

In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus
)
fig.show()

#### Plot noise

In [ ]:
EXPLORATION_PARAMS["weight_noise"]["min"] = 0.0001
EXPLORATION_PARAMS["weight_noise"]["max"] = 1.0
EXPLORATION_PARAMS["tau_OU"]["min"] = 0.4

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=["ou_drift"], 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

for i, row in enumerate(grid):
    for j, parameters in enumerate(row):
        results[i][j] = (results[i][j][0], results[i][j][1] * parameters["weight_noise"])
        

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {BASE_PARAMS[p]["default"]}')
    pprint(f'  Min: {BASE_PARAMS[p]["min"]}')
    pprint(f'  Max: {BASE_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))

In [ ]:
#Lineas a 1 Hz, 10 Hz

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    log_scale=True
)
fig.show()

### Membrane properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Membrane properties'][i:i+2] #+["whatever"]

pprint(f'Membrane properties len: {len(GROUPS["Membrane properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')

In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["g_L"]["min"] = 0.1
# EXPLORATION_PARAMS["g_L"]["max"] = 100.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Adaptation

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Adaptation'][i:i+2] #+["whatever"]

pprint(f'Adaptation len: {len(GROUPS["Adaptation"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')

In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["b_e"]["min"] = 0.0
# EXPLORATION_PARAMS["b_e"]["max"] = 150.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Synaptic properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Synaptic properties'][i:i+2] #+["whatever"]

pprint(f'Synaptic properties len: {len(GROUPS["Synaptic properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["E_e"]["min"] = -20.0
# EXPLORATION_PARAMS["E_e"]["max"] = 20.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Network properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Network properties'][i:i+2] #+["whatever"]

pprint(f'Network properties len: {len(GROUPS["Network properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["N_tot"]["min"] = 1000
# EXPLORATION_PARAMS["N_tot"]["max"] = 50000

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Phenomenological threshold coefficients

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Phenomenological threshold coefficients'] #+["whatever"]
FIRST_PARAM_INDEX = 1
SECOND_PARAM_INDEX = 1


ONLY_TWO_PARAMS = True

pprint(f'Phenomenological threshold coefficients len: {len(GROUPS["Phenomenological threshold coefficients"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# Adjust the coefficient ranges around their defaults.
# EXPLORATION_PARAMS = bounds_from_default(
#     EXPLORATION_PARAMS, PARAMS_TO_EXPLORE, margin=0.10,
# )

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

if ONLY_TWO_PARAMS:
    EXPLORATION_PARAMS = bounds_from_default(
        EXPLORATION_PARAMS, PARAMS_TO_EXPLORE, margin=0.0,
    )
    for name, index in zip(PARAMS_TO_EXPLORE, (FIRST_PARAM_INDEX, SECOND_PARAM_INDEX)):
        default = EXPLORATION_PARAMS[name]['default'][index]
        EXPLORATION_PARAMS[name]['min'][index] = default * 0.9
        EXPLORATION_PARAMS[name]['max'][index] = default * 1.1


GRID_SPACING = 'linear' # 'log' for logarithmic spacing.


USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Inhibitory coupling scaling

**NO**: Se aplica en Zerlaut de primer orden y puede ser muy **relevante**.

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Inhibitory coupling scaling'] #+["whatever"]

pprint(f'Inhibitory coupling scaling len: {len(GROUPS["Inhibitory coupling scaling"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["S_i"]["min"] = 0.0
# EXPLORATION_PARAMS["S_i"]["max"] = 2.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False

# S_i scales long-range coupling in second order; this uncoupled setup has none.


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

#### Se puede simular $S_i$ con **external_in_ex**


$$
\nu_{e\to i}^{\mathrm{input}} = S_i \cdot C_{\mathrm{0}}  + ...
$$

In [ ]:
def simulate_grid_si(grid, stimulus, time_ms, # *,
                  model_name, output_states,
                  integer_parameters, dt_ms, integrator_name,
                  noise_config, noise_seed):
    results = []
    for i, row in enumerate(grid):
        row_results = []
        for j, parameters in enumerate(row):
            pprint(f'Running simulation: {i}, {j}')
            model = build_model(
                parameters, integer_parameters,
                model_class=model_name, variable_of_interest=output_states,
            )


            simulation_stimulus = stimulus
            if 'S_i' in parameters:
                simulation_stimulus = [
                    (name, values * parameters['S_i']
                     if name in ['external_input_in_ex', 'external_input_in_in'] else values)
                    for name, values in stimulus
                ]
            integrator = build_integrator(
                dt_ms, integrator_class=integrator_name,
                noise_config=noise_config,
            )
            simulator = build_single_region_simulator(model, simulation_stimulus, integrator)
            integrator.noise.random_stream.seed(noise_seed)
            time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
            row_results.append((time_sim, raw[:, 0, 0, 0]))
        results.append(row_results)

    return results


In [ ]:

results = simulate_grid_si(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()